# Geração de NetCDF com agentes · Google Colab

Da consulta do dado à geração dele: um **núcleo** comum grava o NetCDF no padrão ARM/CF a
partir de um **plug-in** por aparelho, um **validador** confere o arquivo e um **agente**
(modelo de linguagem) escreve o plug-in de um formato ou aparelho novo, corrigindo até passar
no validador. Detalhes em `docs/geracao_netcdf.md`.

**Como usar:** escolha uma GPU (**Ambiente de execução → Alterar o tipo**; A100 ou L4 de
preferência) e use **Ambiente de execução → Executar tudo**. Não precisa de token do Hugging Face.

O notebook:
1. acha o projeto no seu Google Drive ou, se não achar, baixa o repositório do GitHub;
2. roda os testes automáticos (segundos);
3. converte e valida os seus arquivos do RD-80, **se** houver `.txt` do DISDRODATA numa pasta
   `dados_rd80/` (dentro do projeto ou direto no Meu Drive);
4. confere o circuito do agente com respostas gravadas (tem de dar 7/7);
5. roda o benchmark com um **modelo de verdade** (padrão: Qwen2.5-Coder-7B, aberto):
   7 formatos de arquivo do RD-80, até 5 tentativas cada; leva ~15-40 min;
6. mostra a tabela de resultados e, se o projeto veio do GitHub, guarda uma cópia no Drive.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Projeto
Procura no Meu Drive uma pasta com `geracao_netcdf/agente.py`; se não achar, clona o repositório
público. Para escolher na mão, preencha `PROJETO_FORCADO`. Com `USAR_DRIVE = False` o Drive não é
montado (o projeto vem do GitHub e os resultados ficam só nesta sessão).

In [ ]:
import glob, os, subprocess

USAR_DRIVE = True           # projeto, dados do RD-80 e cópia dos resultados no Google Drive
PROJETO_FORCADO = None      # ex.: "/content/drive/MyDrive/geracao-netcdf-agentes"
REPO_URL = "https://github.com/Nacpa17/geracao-netcdf-agentes.git"
NOME = "geracao-netcdf-agentes"
DRIVE = "/content/drive/MyDrive"


def rodar(cmd, pode_falhar=False):
    """Roda um comando mostrando a saída; interrompe o notebook se falhar."""
    print(">>", cmd, flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1", TQDM_DISABLE="1",
               HF_HUB_DISABLE_PROGRESS_BARS="1")
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, env=env,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linha in p.stdout:
        print(linha, end="")
    if p.wait() != 0 and not pode_falhar:
        raise RuntimeError(f"O comando falhou (código {p.returncode}): {cmd}")
    return p.returncode


if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

achados = []
if USAR_DRIVE and os.path.isdir(DRIVE):
    achados = sorted({os.path.realpath(p).rsplit("/geracao_netcdf/agente.py", 1)[0]
                      for p in glob.glob(f"{DRIVE}/*/geracao_netcdf/agente.py")
                      + glob.glob(f"{DRIVE}/*/*/geracao_netcdf/agente.py")})
if PROJETO_FORCADO:
    PROJETO = PROJETO_FORCADO
elif achados:
    preferidos = [p for p in achados if os.path.basename(p) == NOME]
    PROJETO = (preferidos or achados)[0]
    if len(achados) > 1:
        print("Versões encontradas no Drive:", achados)
else:
    PROJETO = f"/content/{NOME}"
    if not os.path.isdir(PROJETO):
        rodar(f"git clone --depth 1 {REPO_URL} {PROJETO}")
os.chdir(PROJETO)
NO_DRIVE = os.path.realpath(PROJETO).startswith("/content/drive/")
print("Projeto:", PROJETO, "(no Drive)" if NO_DRIVE else "(baixado do GitHub)")

## 3. Bibliotecas
O Colab já tem PyTorch, numpy e pandas. Leva ~1 minuto.

In [ ]:
!pip install -q "transformers==5.20.2" accelerate bitsandbytes netCDF4 pyyaml requests pytest
import torch
print("CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

## 4. Testes automáticos (sem modelo, segundos)

In [ ]:
rodar("python -m pytest -q")

## 5. Converter os seus dados do RD-80 (opcional)
Ponha os `.txt` do DISDRODATA numa pasta `dados_rd80/`, dentro do projeto ou direto no Meu Drive
(ou aponte `DADOS_RD80` para outra pasta). Sai um NetCDF por dia em `saida/rd80/` (dias sem
nenhuma observação não geram arquivo); cada um passa pelo validador do esquema e depois pelo
perfil do padrão ARM 1.3.

In [ ]:
DADOS_RD80 = None   # None = procura dados_rd80/ no projeto e no Meu Drive

candidatos = [DADOS_RD80] if DADOS_RD80 else [os.path.join(PROJETO, "dados_rd80"), f"{DRIVE}/dados_rd80"]
pasta = next((c for c in candidatos if glob.glob(os.path.join(c, "*.txt"))), None)
if pasta:
    rodar(f'python -m geracao_netcdf.gerar --aparelho rd80 --entrada "{pasta}" --saida saida/rd80 --validar')
    rodar("python -m geracao_netcdf.validador saida/rd80/*.nc --aparelho rd80 --padrao arm --curto",
          pode_falhar=True)
else:
    print("Nenhum .txt do RD-80 em", " nem em ".join(candidatos), "— pulei esta etapa.")

## 6. Benchmark com respostas gravadas (confere o circuito)
Escreve o mesmo dado sintético do RD-80 em 7 formatos (TAB/vírgula decimal, `;`, ISO 8601,
hora local, horário de fim, colunas trocadas, cabeçalho comprido) e usa respostas escritas à mão.
Tem de dar **7/7**; em `hora_local` a 1ª resposta esquece o fuso de propósito e os pontos de
controle pegam.

In [ ]:
rodar("python -m geracao_netcdf.bench --llm roteiro:benchmarks/geracao/solucoes --saida benchmarks/results/geracao_roteiro")

## 7. Benchmark com um modelo de verdade
- `MODELO`: `hf:<modelo do Hugging Face>` roda aqui na GPU. Em GPU com menos de 20 GB (T4)
  o modelo é carregado em 4 bits. Também dá para usar uma API: `anthropic:<modelo>` (defina
  `ANTHROPIC_API_KEY` nos *Secrets* do Colab) ou `openai:<modelo>[@url]`.
- `MODO`: `com_exemplo` (o prompt traz o plug-in do RD-80 de outro formato: mede adaptar o
  leitor) ou `do_zero` (só a documentação com as fórmulas: mede escrever tudo).
- O resultado compara cada NetCDF gerado com o gabarito, variável por variável.

In [ ]:
MODELO = "hf:Qwen/Qwen2.5-Coder-7B-Instruct"
MODO = "com_exemplo"            # ou "do_zero"
VARIANTES = ""                  # vazio = todas; ex.: "iso_utc hora_local"
MAX_ITERACOES = 5

extra = ""
if MODELO.startswith("hf:") and torch.cuda.is_available():
    if torch.cuda.get_device_properties(0).total_memory < 20e9:
        extra += " --4bit"
if VARIANTES.strip():
    extra += " --variantes " + VARIANTES
try:                                        # chave de API pelos Secrets do Colab, se houver
    from google.colab import userdata
    for k in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass
SAIDA = f"benchmarks/results/geracao_{MODO}_{MODELO.split(':', 1)[1].replace('/', '_')}"
rodar(f"python -m geracao_netcdf.bench --llm {MODELO} --modo {MODO} --max-iteracoes {MAX_ITERACOES} "
      f"--saida {SAIDA}{extra}", pode_falhar=True)

## 8. Resultado

In [ ]:
from IPython.display import Markdown, display
for r in sorted(glob.glob("benchmarks/results/geracao_*/resumo.md")):
    display(Markdown(open(r, encoding="utf-8").read()))
print("Prompts, respostas, plug-ins e retornos de cada tentativa: benchmarks/results/geracao_*/agente/<formato>/")

## 9. Guardar os resultados
Se o projeto veio do GitHub, ele some quando a sessão do Colab acaba; esta célula copia
`benchmarks/results/` e `saida/` para `Meu Drive/geracao-netcdf-resultados/<data e hora>/`.

In [ ]:
import shutil, time

if NO_DRIVE:
    print("O projeto está no Drive: os resultados já ficaram em", PROJETO)
elif USAR_DRIVE and os.path.isdir(DRIVE):
    destino = f"{DRIVE}/geracao-netcdf-resultados/{time.strftime('%Y-%m-%d_%H%M')}"
    for origem in ("benchmarks/results", "saida"):
        if os.path.isdir(origem):
            shutil.copytree(origem, os.path.join(destino, origem), dirs_exist_ok=True)
    print("Resultados copiados para", destino)
else:
    print("Drive não montado: baixe a pasta benchmarks/results pelo painel de arquivos antes de fechar.")

## 10. Aparelho novo (ex.: Parsivel²)
1. Molde do esquema a partir de um NetCDF de exemplo da ARM do mesmo tipo de aparelho
   (o Parsivel² do ATTO já tem esquema e constantes em `aparelhos_novos/parsivel2/`):
   `python -m geracao_netcdf.molde --de-netcdf exemplo.cdf --saida aparelhos_novos/<nome> --datastream att<sigla>CA.b1 --site att --facility CA --lat -2.1815896 --lon -59.0218639 --alt 59`
2. Ponha 1 a 3 arquivos brutos em `aparelhos_novos/<nome>/amostras/`, descreva o formato em
   `tarefa.yaml` e, se puder, alguns pontos de controle (valores conhecidos).
3. `python -m geracao_netcdf.agente --tarefa aparelhos_novos/<nome>/tarefa.yaml --llm hf:Qwen/Qwen2.5-Coder-7B-Instruct`